In [22]:
def egcd(a: int, b: int):
    """
    Returns (g, x, y) with ax + by = g = gcd(a, b).
    """
    if b == 0:
        return a, 1, 0
    g, x1, y1 = egcd(b, a % b)
    return g, y1, x1 - (a // b) * y1


def modinv(a: int, m: int):
    """
    Returns multiplicative inverse of a modulo m, a⁻¹ (mod m) if it exists
    """
    g, x, _ = egcd(a, m)
    if g == 1:
        return x % m # x may be negative, bring it to the range [0, m‑1]

a = 527_667_751
m = 2_297_634_964
inv = modinv(a, m)
print(f"1.a) The inverse of {a} modulo {m} is {inv}")

p, q = 87_276, 186_270
g, s, t = egcd(p, q)
print(f"1.b) gcd({p}, {q}) = {g}")
print(f"   Bezout identity: {p}({s}) + {q}({t}) = {g}")

# sanity check (optional)
assert p * s + q * t == g

1.a) The inverse of 527667751 modulo 2297634964 is 524038819
1.b) gcd(87276, 186270) = 42
   Bezout identity: 87276(1987) + 186270(-931) = 42


In [23]:
def square_and_multiply(base: int, exponent: int, modulus: int) -> int:
    if modulus == 1:
        return 0                     # everything ≡ 0 (mod 1)

    # Reduce the base once at the start - it keeps the intermediate numbers small.
    base = base % modulus

    result = 1                       # this will accumulate the answer
    e = exponent                     # we will walk through the bits of e

    while e > 0:
        # If the current (least‑significant) bit of the exponent is 1,
        # multiply the current result by the current base.
        if e & 1:                    # test the LSB
            result = (result * base) % modulus

        # Square the base for the next bit (always performed).
        base = (base * base) % modulus

        # Shift the exponent right by one bit → next higher bit.
        e >>= 1

    return result

a = 2_405_210_030          # base
e = 65_537                 # exponent
n = 2_993_725_127          # modulus

res = square_and_multiply(a, e, n)
print("Square-and-Multiply result :", res)

# Verify against Python's built‑in pow
builtin = pow(a, e, n)
print("Python built-in pow result :", builtin)

assert res == builtin

Square-and-Multiply result : 2453137349
Python built-in pow result : 2453137349


In [24]:
def crt(remainders, moduli):
    # product of all moduli
    M = 1
    for m in moduli:
        M *= m

    # accumulate the solution
    x = 0
    for r, m in zip(remainders, moduli):
        N_i = M // m                 # N_i = M / m_i
        inv = modinv(N_i, m)         # N_i^{-1} (mod m_i)
        term = (r * N_i * inv) % M   # we can reduce each term modulo M
        x = (x + term) % M

    return x, M


a = [29683, 144995, 136776]         # remainders
m = [163659, 146921, 193331]        # moduli (pairwise coprime)

solution, modulus = crt(a, m)

print("Solution of the CRT system")
print("---------------------------")
print(f"x ≡ {solution} (mod {modulus})")
print("\nVerification:")
for i, (ri, mi) in enumerate(zip(a, m), 1):
    print(f"  x ≡ {solution % mi} (mod {mi})   →   expected {ri}")

Solution of the CRT system
---------------------------
x ≡ 6903361 (mod 4648633056670809)

Verification:
  x ≡ 29683 (mod 163659)   →   expected 29683
  x ≡ 144995 (mod 146921)   →   expected 144995
  x ≡ 136776 (mod 193331)   →   expected 136776


In [25]:
import math
import sys

n1 = int(
    "41586933446220082872168792411016360390278592380680477028872096588587928594989"
)
n2 = int(
    "5010609569443118167691454968557827442659503676125410711002434917434601365503"
)

# Compute the common factor (the shared prime)
p = math.gcd(n1, n2)

if p == 1:
    print("The two moduli are coprime, no common prime was found.")
    sys.exit(0)

# Recover the remaining private primes
q1 = n1 // p
q2 = n2 // p

#  Print the results (the three primes are guaranteed to be prime)
print("\n===  RSA factorisation obtained from the two public keys  ===")
print(f"shared prime   p = {p}")
print(f"n1 = p * q1  →  q1 = {q1}")
print(f"n2 = p * q2  →  q2 = {q2}")


===  RSA factorisation obtained from the two public keys  ===
shared prime   p = 218850498245032537348796224557686828023
n1 = p * q1  →  q1 = 190024394642492078289673158706424130043
n2 = p * q2  →  q2 = 22895125255018005641165540395055880761


In [26]:

p = 853
q = 223
n = p * q                     # modulus
phi = (p - 1) * (q - 1)       # phi(n)

e = 65537                     # public exponent (chosen coprime to phi)
# compute the private exponent d = e^{-1} (mod phi)
g, x, _ = egcd(e, phi)

d = x % phi                    # private exponent

print(f"Public key  (e, n) = ({e}, {n})")
print(f"Private key (d, n) = ({d}, {n})\n")


# Helper: convert a 3‑letter block to an integer (base‑26)
ALPHABET = {chr(i + ord('A')): i for i in range(26)}   # A:0 … Z:25

def block_to_int(block: str) -> int:
    """block must be exactly three upper-case letters"""
    l1, l2, l3 = (ALPHABET[ch] for ch in block)
    return l1 * 26 * 26 + l2 * 26 + l3

plaintext = "CYBERSECURITYISFUN" # plaintext is already a multiple of 3

blocks = [plaintext[i:i+3] for i in range(0, len(plaintext), 3)]
msg_numbers = [block_to_int(b) for b in blocks]

print("Plain-text blocks and their numeric values:")
for b, m in zip(blocks, msg_numbers):
    print(f"  {b} → {m}")
print()

# Encryption (square‑and‑multiply via pow)
cipher = [pow(m, e, n) for m in msg_numbers]

print("Cipher‑text (one integer per block):")
print(cipher)

Public key  (e, n) = (65537, 190219)
Private key (d, n) = (40985, 190219)

Plain-text blocks and their numeric values:
  CYB → 1977
  ERS → 3164
  ECU → 2776
  RIT → 11719
  YIS → 16450
  FUN → 3913

Cipher‑text (one integer per block):
[188448, 180241, 148736, 7364, 134589, 49664]
